# Meeting Memory Agent: Colab demo

An agent that turns 138 real meeting transcripts (AMI corpus) into searchable memory, answers questions with **word-for-word citations**, and knows **what changed and what is current**.

Repo: https://github.com/raheemsultan337-ship-it/meeting-memory-agent · Write-up: `WRITEUP.md` · Eval: `eval/report.md`

**Runtime → Run all.** Sections 1-5 need **no API key**: they replay from the shipped memory store (`memory.db`) and the cache of every model response (`cache.db`). Section 6 asks a new question live and needs a free Gemini key. Section 7 (optional) reproduces and validates `answers.json` from Catalist's data files.

## 1. Setup (about 1 minute)

If pip prints a red warning about package versions, or Colab offers to restart the session, it is safe to ignore (or click *Restart* and run this cell again).

In [ ]:
import os, json, glob
os.chdir("/content")
if not os.path.exists("meeting-memory-agent"):          # safe to re-run
    !git clone -q https://github.com/raheemsultan337-ship-it/meeting-memory-agent.git
%cd /content/meeting-memory-agent
# Quiet install. pip may still warn about Colab's preinstalled packages; that is harmless,
# because every project command below runs in a fresh `!python` process.
!pip install -q google-genai==3.6.1 rapidfuzz python-dotenv tzdata 2>&1 | grep -v -i "dependency resolver\|incompatible" || true
os.environ["PYTHONIOENCODING"] = "utf-8"
print("ready:", os.getcwd())

## 2. The submission: `answers.json`
25 answers, each with 1-3 citations (meeting id + exact words from one speaker turn), plus the brief for Team TS3012.

In [ ]:
data = json.load(open("answers.json", encoding="utf-8"))
for a in data["answers"][:5]:
    print(a["question_id"] + ": " + a["answer"])
    for c in a["citations"]:
        print("    [" + c["meeting_id"] + "] '" + c["quote"] + "'")
print("...")
print()
print("=== BRIEF (first lines) ===")
for line in data["brief"].splitlines()[:12]:
    print(line)

## 3. Memory over time: how the agent knows what is current
Per team and topic: history oldest-first (each step with the speaker's exact words) and the **current** state. Here the case material **CHANGED**: hard plastic (a) → titanium back (c) → titanium-coloured plastic (d).

In [ ]:
!python timeline.py --show TS3009

## 4. Why hybrid search
Keyword search (BM25) matches words, embeddings match meaning, and Reciprocal Rank Fusion combines the two rankings. `ranks` shows which method found each passage.

In [ ]:
!python search.py --compare "how did they stop the remote getting lost?" ES2005

## 5. The agent, step by step (replayed from cache, no key)
Watch the loop: it starts from the team timeline, calls tools (`search`, `read_turns`, `find_facts`, `get_timeline`), then `submit_answer`. The **citation gate** (plain code) checks every quote word for word and sends problems back to the model.

In [ ]:
!python agent.py ask "Team TS3009 planned a titanium back for their remote. Who first proposed replacing the titanium, what did they propose instead, and was the titanium kept in the end?"

## 6. Ask your own question (needs a free Gemini key)

**Step 1: get a key.** Free at https://aistudio.google.com/apikey (Create API key).

**Step 2: give it to the notebook.** Either:
- click the **🔑 key icon** in Colab's left sidebar → **Add new secret** → Name `GEMINI_API_KEY`, Value: your key → turn on **Notebook access**, or
- skip that and paste the key into the box that appears when you run the next cell.

**Step 3: run the key cell below.** It should print `key set: True`.

**Step 4: edit the `question = "..."` line** in the cell after it, then press **Shift+Enter**. You'll see each tool call the agent makes, then the **ANSWER** and its verified **citations** (about 10-30 seconds).

**Rules for the question**
- It must name a team: `ES2002`-`ES2016`, `IS1000`-`IS1009` or `TS3003`-`TS3012`, e.g. *"In Team IS1009, what did they decide about the case colour?"*
- Don't use double quotes (`"`) inside the question; single quotes are fine.

**Free-tier notes:** about 6 model calls per question (flash-lite allows 500/day per Google project). Lines like `503, waiting 4s` mean Google is busy and the retry logic is working. If the day's quota is used up, it prints *"Can't answer this new question right now: ...quota..."*. Quotas reset at midnight Pacific, or create a key in a new project in AI Studio for a fresh quota.

In [ ]:
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")
print("key set:", bool(os.environ.get("GEMINI_API_KEY")))

In [ ]:
question = "Team ES2008 wanted a feature to stop the remote getting lost. What happened to that feature, and who objected?"  # edit me
!python agent.py ask "{question}"

## 7. Optional: reproduce and validate `answers.json`
Needs Catalist's data folder (`questions.json` + `transcripts/`), which is not in the repo. Upload it as a zip, and the pipeline replays every agent step from cache (**zero API calls**, no key needed), rebuilds `answers.json`, and checks it against every submission rule. Then the 39 unit tests run.

In [ ]:
from google.colab import files
up = files.upload()                      # choose Catalist_AI_Intern_Build_Test_Data.zip
zipname = next(iter(up))
!unzip -q -o "{zipname}" -d /content/data
import glob
qpath = glob.glob("/content/data/**/questions.json", recursive=True)[0]
os.environ["DATA_DIR"] = os.path.dirname(qpath)
print("DATA_DIR =", os.environ["DATA_DIR"])

In [ ]:
!cp answers.json answers.submitted.json
!python run.py answer && python run.py brief && python run.py assemble
same = json.load(open("answers.json", encoding="utf-8")) == json.load(open("answers.submitted.json", encoding="utf-8"))
print()
print("Reproduced answers.json identical to the submitted one:", same)

In [ ]:
!pip install -q pytest && python -m pytest -q